# Análisis Exploratorio de Datos (EDA) - The Last Mile Gap

**Objetivo:** Documentar la exploración de los datos de Olist, validar la calidad del dataset y generar las visualizaciones base para responder a las 5 preguntas del Business Case.

**Autor:** Julio Sosa Angeles

In [1]:
import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

# Cargar entorno
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# Cargar la vista principal
query = "SELECT * FROM vw_order_delivery_metrics WHERE review_score IS NOT NULL"
df = pd.read_sql(query, engine)
print(f"Filas cargadas: {len(df)}")
df.head()

Filas cargadas: 96353


,order_id,customer_id,customer_state,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,review_score,dias_diferencia,is_late
0,73fc7af87114b39712e6da79b0a377eb,41dcb106f807e993532d446263290104,SP,2018-01-11 15:30:49,2018-01-17 18:42:41,2018-02-02,4,-15.0,0
1,8e6bfb81e283fa7e4f11123a3fb894f1,5986b333ca0d44534a156a52a8e33a83,SC,2018-02-10 10:59:03,2018-02-28 16:33:35,2018-03-09,5,-8.0,0
2,b18dcdf73be66366873cd26c5724d1dc,eecafc3ff695f031bfe354a9fff9d437,SP,2018-04-06 22:18:54,2018-04-12 17:17:53,2018-05-03,1,-20.0,0
3,e48aa0d2dcec3a2e87348811bcfdf22b,840be0730010e9580efd289bc9f37706,SP,2017-06-30 15:38:46,2017-07-15 12:57:24,2017-08-03,5,-18.0,0
4,c31a859e34e3adac22f376954e19b39d,98f485515e575607a8003b17e91990bd,RJ,2018-08-07 23:12:29,2018-08-13 18:08:28,2018-10-10,5,-57.0,0


## Pregunta 1: ¿Cuál es la relación entre el retraso y la reseña?
Vamos a agrupar por estado de retraso y calcular el promedio de reseña.

In [5]:
df_q1 = df.groupby('is_late')['review_score'].mean().reset_index()
df_q1['Estado'] = df_q1['is_late'].map({0: 'A tiempo', 1: 'Tardío'})

fig1 = px.bar(df_q1, x='Estado', y='review_score', color='Estado', text_auto='.2f',
              title="Impacto del retraso en la satisfacción",
              color_discrete_sequence=['#2ca02c', '#d62728'])
fig1.show()

## Pregunta 2: ¿Qué estados concentran la mayor proporción de retrasos?
Filtramos estados con más de 100 pedidos para evitar sesgos por bajo volumen.

In [4]:
df_q2 = df.groupby('customer_state')['is_late'].agg(['count', 'mean']).reset_index()
df_q2 = df_q2[df_q2['count'] > 100]
df_q2['porcentaje_tardios'] = df_q2['mean'] * 100
df_q2 = df_q2.sort_values(by='porcentaje_tardios', ascending=False).head(10)

fig2 = px.bar(df_q2, x='customer_state', y='porcentaje_tardios', 
              title="Top 10 Estados con mayor % de retrasos",
              labels={'porcentaje_tardios': '% Retrasos', 'customer_state': 'Estado'},
              color='porcentaje_tardios', color_continuous_scale='YlOrRd')
fig2.show()

## Pregunta 3: ¿Qué sellers presentan la peor performance?

In [7]:
# Traemos los datos de sellers desde la base de datos
query_sellers = """
    SELECT 
        LEFT(s.seller_id, 8) AS seller_id,
        COUNT(DISTINCT oi.order_id) AS total_pedidos,
        ROUND(AVG(r.review_score), 2) AS resena_promedio
    FROM order_items oi
    JOIN sellers s ON oi.seller_id = s.seller_id
    JOIN orders o ON oi.order_id = o.order_id
    LEFT JOIN order_reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY s.seller_id
    HAVING COUNT(DISTINCT oi.order_id) > 50
    ORDER BY resena_promedio ASC
    LIMIT 10;
"""
with engine.connect() as conn:
    df_sell = pd.read_sql(query_sellers, conn)
df_sell


,seller_id,total_pedidos,resena_promedio
0,1ca7077d,108,2.27
1,2eb70248,187,2.81
2,972d0f9c,81,2.96
3,a49928bc,96,2.97
4,8e6d7754,84,3.03
5,bbad7e51,68,3.05
6,54965bbe,73,3.07
7,5058e8c1,62,3.08
8,8444e55c,92,3.16
9,b2479f94,101,3.32


## Pregunta 4: ¿Qué categorías tienen la peor reseña y si está asociada a retrasos?

In [8]:
# Consulta de categorías (para el gráfico de dispersión de la US-16)
query_cat = """
    SELECT 
        COALESCE(pct.product_category_name_english, p.product_category_name) AS categoria,
        ROUND(AVG(r.review_score), 2) AS resena_promedio,
        ROUND((SUM(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END) * 100.0 / COUNT(DISTINCT o.order_id)), 2) AS porcentaje_tardios
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    LEFT JOIN product_category_name_translation pct ON p.product_category_name = pct.product_category_name
    JOIN orders o ON oi.order_id = o.order_id
    LEFT JOIN order_reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered' AND p.product_category_name IS NOT NULL
    GROUP BY p.product_category_name, pct.product_category_name_english
    HAVING COUNT(DISTINCT o.order_id) > 100
    ORDER BY resena_promedio ASC
    LIMIT 15;
"""
with engine.connect() as conn:
    df_cat = pd.read_sql(query_cat, conn)

# Gráfico de dispersión
fig4 = px.scatter(df_cat, x='porcentaje_tardios', y='resena_promedio',
                  hover_name='categoria', title="Categorías: Retraso vs. Reseña",
                  labels={'porcentaje_tardios': '% Retrasos', 'resena_promedio': 'Reseña Promedio'},
                  color_discrete_sequence=['#d62728'])
fig4.show()

## Pregunta 5: ¿El retraso afecta por igual a pedidos con uno o varios ítems?
*Conclusión:* Tras el análisis en SQL (ver `docs/recommendations.md`), se determinó que el retraso es transversal. Los pedidos de 1 ítem tienen 8.29% de retraso, mientras que los de 4+ ítems tienen 6.19%. La diferencia no es significativa como para sesgar el análisis.

## Calidad del dato
*Ver reporte completo en `docs/data_quality.md`.*
- 2.98% de los pedidos no tienen fecha de entrega (fueron excluidos de la vista principal).
- 0 duplicados en `orders` y `order_items`.
- 0 fechas inconsistentes (entregas antes de la compra).

## Conclusiones del EDA
1. Se confirma la hipótesis del Business Case: el retraso reduce la reseña de 4.29 a 2.57.
2. Existe una fuerte correlación geográfica: los estados del Nordeste (AL, MA, PI) son los más afectados.
3. La cantidad de ítems por pedido no sesga los resultados de retraso.
4. Los datos están limpios (0 duplicados) y los nulos de fecha de entrega (2.98%) fueron excluidos correctamente.